In [ ]:
from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
import logging
import sys

logging.basicConfig(
    stream=sys.stdout, level=logging.INFO
)  # logging.DEBUG for more verbose output
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

In [ ]:
import os
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from dotenv import load_dotenv
from datetime import datetime
import logging


os.environ["AZURE_SUBSCRIPTION_ID"]='*****'
# Set Azure environment variables
os.environ["AZURE_RESOURCE_GROUP_NAME"] = "*****"
os.environ["AZURE_PROJECT_NAME"] = "*****"
os.environ["AZURE_PROJECT_ENDPOINT"] = "*****"
os.environ["AZURE_DEPLOYMENT_ID"] = "gpt-4o"#"o1"
os.environ["AZURE_API_VERSION"] = "*****"
os.environ["AZURE_OPENAI_ENDPOINT"] = "*****"
os.environ["AZURE_OPENAI_ACCOUNT_NAME"] = "*****"


load_dotenv()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

required_vars = [
    "AZURE_OPENAI_ENDPOINT", 
     "AZURE_DEPLOYMENT_ID",
    "AZURE_SUBSCRIPTION_ID","AZURE_API_VERSION","AZURE_PROJECT_ENDPOINT",
    "AZURE_PROJECT_NAME"
]

for var in required_vars:
    if not os.getenv(var):
        logger.error(f"Missing required environment variable: {var}")
        raise ValueError(f"Missing required environment variable: {var}")
        
from azure.identity import DefaultAzureCredential
from llama_index.llms.azure_openai import AzureOpenAI

# Step 1: Get AAD token
credential = DefaultAzureCredential()
token = credential.get_token("https://cognitiveservices.azure.com/.default")

In [ ]:
import pandas as pd
import json
import requests
import openai

import os
from dotenv import load_dotenv
import pandas as pd

from pathlib import Path
import streamlit as st
from openai import OpenAI
from llama_index.core.query_engine import PandasQueryEngine
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core.program import LLMTextCompletionProgram
from pydantic import BaseModel, Field
from llama_index.core.program import LLMTextCompletionProgram
from sqlalchemy import (create_engine, MetaData, Table, Column, String, Integer, text,inspect)
import re
from llama_index.core.objects import(SQLTableNodeMapping, ObjectIndex, SQLTableSchema)
from llama_index.core import SQLDatabase, VectorStoreIndex, load_index_from_storage
from llama_index.core.retrievers import SQLRetriever
from typing import List, Dict
from llama_index.core.query_pipeline import (FnComponent, QueryPipeline as QP, Link, InputComponent, CustomQueryComponent)
from llama_index.core.prompts.default_prompts import DEFAULT_TEXT_TO_SQL_PROMPT
from llama_index.core import PromptTemplate
from llama_index.core.base.llms.types import ChatResponse
from llama_index.core.service_context import ServiceContext
from llama_index.core.schema import TextNode
from llama_index.core.storage import StorageContext
# from llama_index.llms import OpenAi

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# Use BGE (BAAI General Embedding) model - open source, high performance
# Options: "BAAI/bge-small-en-v1.5", "BAAI/bge-base-en-v1.5", "BAAI/bge-large-en-v1.5"
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-large-en-v1.5",  # Best quality, larger model
    device="cuda",  # Use "cuda" for GPU, "cpu" for CPU
    cache_folder="./model_cache"  # Cache downloaded models locally
)

In [ ]:
from llama_index.llms.azure_openai import AzureOpenAI

# ---- Backbone LLM configuration (parameterized) ----
# Change these to switch the GPT-4o deployment without editing the rest of the code.
LLM_MODEL = os.getenv("AZURE_OPENAI_MODEL", "gpt-4o")        # model name
LLM_DEPLOYMENT = os.getenv("AZURE_DEPLOYMENT_ID", "gpt-4o")  # Azure deployment name
LLM_TEMPERATURE = 0  # paper: decoding temperature fixed at 0 for deterministic, reproducible output

Settings.llm = AzureOpenAI(
    model=LLM_MODEL,
    deployment_name=LLM_DEPLOYMENT,
    temperature=LLM_TEMPERATURE,
    api_key=token.token,
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version=os.environ["AZURE_API_VERSION"],  # e.g., 2024-02-01
)

In [ ]:
# --- Auto-load table descriptions and schema from database ---
import sys
from pathlib import Path

# Add utils directory to Python path
utils_path = Path(".").resolve() / "utils"
if str(utils_path) not in sys.path:
    sys.path.insert(0, str(utils_path))

from db_schema_loader import auto_load_database
import pandas as pd
from sqlalchemy import create_engine

# Automatically discover and load:
# 1. Database file (my_database_mimic.db)
# 2. Table descriptions (sqlite_table_descriptions.json - generated by 2_generate_descriptions.py)
# 3. Table summaries (tableinfos/ directory)
# 4. Column names from database
table_names, table_names_and_infos, table_names_and_columns = auto_load_database(
    search_dir="../",  # Search in parent directory for data files
)

# Create database engine for querying
db_path = "../my_database_mimic.db"  # Database in parent directory
SQL_TIMEOUT_SECONDS = 120  # paper: SQL execution guarded with a 120-second timeout
engine = create_engine(
    f"sqlite:///{db_path}",
    connect_args={"timeout": SQL_TIMEOUT_SECONDS},
)

# Optional: Load DataFrames for each table if needed
# names_and_frames = {table_name: pd.read_sql_table(table_name, engine) for table_name in table_names}

In [ ]:
def sanitize_column_name(col_name):
    return re.sub(r"\W+", "_", col_name)

def create_table_from_dataframe(df: pd.DataFrame, table_name: str, engine, metadata_obj):
    sanitized_columns = {col: sanitize_column_name(col) for col in df.columns}
    df = df.rename(columns=sanitized_columns)

    columns=[
        Column(col, String if dtype == "object" else Integer)
        for col, dtype in zip(df.columns, df.dtypes)
    ]

    table = Table(table_name, metadata_obj, *columns)

    metadata_obj.create_all(engine)

    with engine.connect() as conn:
        for _, row in df.iterrows():
            insert_stmt = table.insert().values(**row.to_dict())
            conn.execute(insert_stmt)
        conn.commit()

# Reuse the 120-second SQL timeout defined above
engine = create_engine(
    f"sqlite:///{db_path}",
    connect_args={"timeout": SQL_TIMEOUT_SECONDS},
)
metadata_obj = MetaData()

# If you need to (re)create tables from DataFrames, uncomment below:
# for table_name, df in names_and_frames.items():
#     print(f"Creating table: {table_name}")
#     create_table_from_dataframe(df, table_name, engine, metadata_obj)

metadata_obj.reflect(bind=engine)

existing_tables = metadata_obj.tables.keys()
print(f"Existing tables: {list(existing_tables)}")

from llama_index.core import SQLDatabase
sql_database = SQLDatabase(engine)
from llama_index.core.objects import SQLTableNodeMapping, SQLTableSchema
table_node_mapping = SQLTableNodeMapping(sql_database)
table_schema_objs = [SQLTableSchema(table_name=t, context_str=table_names_and_infos[t]) for t in table_names]

table_column_schema = [SQLTableSchema(table_name=t, context_str=str(table_names_and_columns[t])) for t in table_names]

from llama_index.core.retrievers import SQLRetriever
sql_retriever = SQLRetriever(sql_database)

In [ ]:
# from sqlalchemy import text, inspect

# def get_existing_tables(engine):
#     inspector = inspect(engine)
#     # Normalize to upper-case to match OMOP names (SQLite is case-insensitive for identifiers)
#     return {name.upper() for name in inspector.get_table_names()}

# def validate_and_load_omop_tables(engine, names_and_frames, expected_cols, if_exists='replace'):
#     """
#     Validate DataFrame columns against expected OMOP schemas and load into DB.
#     - if_exists: 'fail' | 'replace' | 'append'
#     """
#     import pandas as pd

#     for tbl, df in names_and_frames.items():
#         if df is None:
#             print(f"[SKIP] {tbl}: DataFrame is None")
#             continue
#         # Ensure DataFrame columns are upper-cased to match OMOP schema
#         df = df.copy()
#         df.columns = [c.upper() for c in df.columns]

#         exp = expected_cols.get(tbl, None)
#         if exp is None:
#             print(f"[WARN] {tbl}: No expected schema provided; loading as-is.")
#         else:
#             missing = [c for c in exp if c not in df.columns]
#             if missing:
#                 print(f"[WARN] {tbl}: Missing expected columns {missing}. Will still load available columns.")
#             # Keep only known columns (order preserved)
#             keep = [c for c in exp if c in df.columns]
#             # Also keep any extra columns to avoid accidental data loss
#             extras = [c for c in df.columns if c not in exp]
#             df = df[keep + extras]

#         print(f"[LOAD] {tbl}: rows={len(df)}, cols={len(df.columns)}, if_exists={if_exists}")
#         df.to_sql(tbl, engine, index=False, if_exists=if_exists)  # pandas will create tables as needed

# def create_omop_indexes(engine):
#     """
#     Create useful indexes for OMOP core tables. SQLite syntax used.
#     """
#     existing = get_existing_tables(engine)

#     # Base indexes by table -> list of column lists (single or composite)
#     index_plan = {
#         'PERSON': [['PERSON_ID']],
#         'CONCEPT': [['CONCEPT_ID'], ['CONCEPT_NAME']],
#         'VISIT_OCCURRENCE': [['VISIT_OCCURRENCE_ID'], ['PERSON_ID'], ['VISIT_CONCEPT_ID']],
#         'CONDITION_OCCURRENCE': [['CONDITION_OCCURRENCE_ID'], ['PERSON_ID'], ['CONDITION_CONCEPT_ID'], ['VISIT_OCCURRENCE_ID'], ['PERSON_ID','VISIT_OCCURRENCE_ID']],
#         'DRUG_EXPOSURE': [['DRUG_EXPOSURE_ID'], ['PERSON_ID'], ['DRUG_CONCEPT_ID'], ['VISIT_OCCURRENCE_ID'], ['PERSON_ID','VISIT_OCCURRENCE_ID']],
#         'MEASUREMENT': [['MEASUREMENT_ID'], ['PERSON_ID'], ['MEASUREMENT_CONCEPT_ID'], ['VISIT_OCCURRENCE_ID'], ['PERSON_ID','MEASUREMENT_CONCEPT_ID']],
#         'OBSERVATION': [['OBSERVATION_ID'], ['PERSON_ID'], ['OBSERVATION_CONCEPT_ID'], ['VISIT_OCCURRENCE_ID']],
#         'PROCEDURE_OCCURRENCE': [['PROCEDURE_OCCURRENCE_ID'], ['PERSON_ID'], ['PROCEDURE_CONCEPT_ID'], ['VISIT_OCCURRENCE_ID']]
#     }

#     # Build CREATE INDEX statements only for existing tables
#     stmts = []
#     for tbl, cols_list in index_plan.items():
#         if tbl not in existing:
#             continue
#         for cols in cols_list:
#             col_part = "_".join(cols).lower()
#             idx_name = f"idx_{tbl.lower()}_{col_part}"
#             col_sql = ", ".join(cols)
#             stmts.append(f"CREATE INDEX IF NOT EXISTS {idx_name} ON {tbl}({col_sql})")

#     try:
#         with engine.connect() as conn:
#             for s in stmts:
#                 print(f"Executing: {s}")
#                 conn.execute(text(s))
#             conn.commit()
#         print("All OMOP indexes created successfully.")
#     except Exception as e:
#         print(f"Error creating OMOP indexes: {e}")

# def note_on_foreign_keys_sqlite():
#     print("Note: SQLite can enforce foreign keys only if defined at table-creation time.")
#     print("To add FK constraints (e.g., CHILD.PERSON_ID -> PERSON.PERSON_ID) after creation, you must:")
#     print("  1) CREATE new tables with FOREIGN KEY clauses; 2) copy data; 3) drop old tables; 4) rename new tables.")
#     print("Also ensure PRAGMA foreign_keys=ON for enforcement during connections.")

# # ---- Usage example ----
# # 1) Enable FK checks (enforced only for tables that were created with FK clauses)
# with engine.connect() as connection:
#     connection.execute(text("PRAGMA foreign_keys = ON"))
#     connection.commit()

# # 2) Validate and load your DataFrames into OMOP tables
# validate_and_load_omop_tables(engine, names_and_frames, table_names_and_columns, if_exists='replace')

# # 3) Create helpful OMOP indexes
# create_omop_indexes(engine)

# # 4) Reminder about FKs in SQLite
# note_on_foreign_keys_sqlite()

In [ ]:
def index_all_tables(sql_database: SQLDatabase, table_index_dir: str="table_index_dir") -> VectorStoreIndex:
    """Index all tables."""

    if not Path(table_index_dir).exists():
        os.mkdir(table_index_dir)

    index_dir = f"{table_index_dir}/table_all"

    if not os.path.exists(index_dir):
        # First run: build new index
        nodes = [
            TextNode(text='Table_name: '+ str(t) + ';\nDescription: ' + str(table_names_and_infos[t]))
            for t in table_names_and_infos
        ]
        index = VectorStoreIndex(
            nodes,
            callback_manager=Settings.callback_manager,
            llm=Settings.llm,
            embed_model=Settings.embed_model
        )
        index.set_index_id("vector_index")
        index.storage_context.persist(index_dir)
    else:
        # Subsequent runs: load directly from storage
        storage_context = StorageContext.from_defaults(persist_dir=index_dir)
        index = load_index_from_storage(
            storage_context,
            index_id="vector_index",
            llm=Settings.llm,
            embed_model=Settings.embed_model
        )

    return index

vector_index = index_all_tables(sql_database)
obj_retriever = vector_index.as_retriever(similarity_top_k=10)

In [ ]:
def get_table_context_str(selected_tables):
    """Get table and columns"""

    context_strs = []
    for content in selected_tables:
        table_name = content.get_content().split(';\n')[0].split('Table_name: ')[1]
        table_info = sql_database.get_single_table_info(
            table_name
        )
        if content.get_content():
            table_opt_context = " The table is: "
            table_opt_context += table_names_and_infos.get(table_name, "")
            table_info += table_opt_context
#        sample_row = ''
        inspector = inspect(engine)
        columns = table_names_and_columns.get(table_name, [column['name'] for column in inspector.get_columns(table_name)])
        
        query = f"SELECT * FROM {table_name} LIMIT 1"
        with engine.connect() as conn:
            result = conn.execute(text(query)).fetchone()
            
        table_info += str(dict(zip(columns, result)) if result else {})
        context_strs.append(table_info)

    return "\n\n".join(context_strs)

table_parser_component = FnComponent(fn=get_table_context_str)

In [ ]:
from llama_index.core import VectorStoreIndex, load_index_from_storage, Settings
from llama_index.core.schema import TextNode
from llama_index.core.storage import StorageContext
from llama_index.core.node_parser import SentenceSplitter
from sqlalchemy import text
import os
import shutil
import traceback

# Directory to store patient indexes
INDEX_DIR = "patient_indexes"
os.makedirs(INDEX_DIR, exist_ok=True)

# paper: notes chunked into 256-token chunks with 32-token overlap, sentence-aware
_note_splitter = SentenceSplitter(chunk_size=256, chunk_overlap=32)

def _build_note_index(text_nodes):
    chunks = _note_splitter.get_nodes_from_documents(text_nodes)  # keeps note_time metadata per chunk
    # paper: concatenate each chunk with its timestamp so temporal context is embedded/retrieved
    for c in chunks:
        t = (c.metadata or {}).get("note_time")
        if t:
            c.text = f"[{t}] {c.text}"
    return VectorStoreIndex(chunks, embed_model=Settings.embed_model)

class CustomTextNode(TextNode):
    def get_doc_id(self):
        return self.id_

def _fetch_notes_with_time(engine, person_id):
    queries = [
        # OMOP CDM
        "SELECT NOTE_TEXT, NOTE_DATETIME AS NOTE_TIME FROM NOTE WHERE PERSON_ID = :pid",
        "SELECT NOTE_TEXT FROM NOTE WHERE PERSON_ID = :pid",
        # MIMIC-III NOTEEVENTS
        "SELECT TEXT AS NOTE_TEXT, CHARTTIME AS NOTE_TIME FROM NOTEEVENTS WHERE SUBJECT_ID = :pid",
        "SELECT TEXT AS NOTE_TEXT, CHARTDATE AS NOTE_TIME FROM NOTEEVENTS WHERE SUBJECT_ID = :pid",
        "SELECT TEXT AS NOTE_TEXT FROM NOTEEVENTS WHERE SUBJECT_ID = :pid",
    ]
    with engine.connect() as conn:
        for q in queries:
            try:
                rows = conn.execute(text(q), {"pid": person_id}).mappings()
                out = []
                for r in rows:
                    if "NOTE_TIME" in r:
                        out.append((r["NOTE_TEXT"], r["NOTE_TIME"]))
                    else:
                        out.append((r["NOTE_TEXT"], None))
                if out:
                    return out
            except Exception:
                # Continue trying next query format
                continue
    return []

def retrieve_notes(inputs, structured_results, top_k: int = 5):
    """
    Retrieve the top-K most relevant note chunks for a given patient_id, with index persistence.
    """
    try:
        # Step 1: Extract person_id using LLM
        prompt = f"Please extract patient id from the question, only return the id. Question: {inputs}"
        person_id = Settings.llm.complete(prompt).text.strip()

        # Validate person_id
        if not person_id.isdigit():
            return "Invalid patient ID extracted."
        person_id = int(person_id)

        # Step 2: Check if the index for this patient already exists on disk
        index_path = os.path.join(INDEX_DIR, f"patient_{person_id}_index")
        if os.path.exists(index_path):
            print(f"Loading index from {index_path}")
            try:
                storage_context = StorageContext.from_defaults(persist_dir=index_path)
                index = load_index_from_storage(storage_context)
            except Exception as e:
                print(f"Error loading index: {e}. Deleting corrupted index.")
                shutil.rmtree(index_path)
                print(f"Deleted corrupted index at {index_path}. Recreating index.")

                notes = _fetch_notes_with_time(engine, person_id)
                if not notes:
                    return "No notes found for this patient."

                text_nodes = [
                    CustomTextNode(
                        text=note_text,
                        id_=f"note_{i}",
                        metadata={"note_time": str(note_time) if note_time is not None else None},
                    )
                    for i, (note_text, note_time) in enumerate(notes)
                ]
                index = _build_note_index(text_nodes)
                index.storage_context.persist(persist_dir=index_path)
        else:
            print(f"Index not found at {index_path}, creating a new one.")

            notes = _fetch_notes_with_time(engine, person_id)
            if not notes:
                return "No notes found for this patient."

            text_nodes = [
                CustomTextNode(
                    text=note_text,
                    id_=f"note_{i}",
                    metadata={"note_time": str(note_time) if note_time is not None else None},
                )
                for i, (note_text, note_time) in enumerate(notes)
            ]

            try:
                index = _build_note_index(text_nodes)
            except Exception as e:
                print("Error creating VectorStoreIndex:\n" + traceback.format_exc())
                return f"Error creating VectorStoreIndex: {e}"

            try:
                index.storage_context.persist(persist_dir=index_path)
                print(f"Index successfully persisted at {index_path}")
            except Exception as e:
                print("Error persisting index:\n" + traceback.format_exc())
                return "An error occurred while persisting the index."

        # Step 3: Retrieve the top-K relevant chunks using the input question + structured evidence
        retriever = index.as_retriever(similarity_top_k=top_k)
        retrieved_nodes = retriever.retrieve(inputs + '\n' + str(structured_results)[:500])

        if not retrieved_nodes:
            return "No relevant notes found."

        # paper: feed the top-K retrieved chunks (each already carries its timestamp) to synthesis
        passages = []
        for n in retrieved_nodes:
            txt = getattr(n, "text", None)
            if txt is None and hasattr(n, "node"):
                try:
                    txt = n.node.get_text()
                except Exception:
                    txt = str(n)
            if txt:
                passages.append(txt)

        return "\n\n".join(passages) if passages else "No relevant notes found."

    except Exception as e:
        # Print full stack trace, return brief message
        print("Error retrieving notes:\n" + traceback.format_exc())
        return f"An error occurred while retrieving notes: {e}"

note_retriever_component = FnComponent(fn=retrieve_notes)

In [ ]:
import json
from llama_index.core.llms import ChatResponse  # Adjust according to your version

def parse_response_to_sql(response: ChatResponse) -> str:
    """Extract SQL from GPT-4o-style markdown JSON code block."""
    
    response_text = response.message.content.strip()

    # Step 1: Extract ```json ... ``` block from markdown
    if "```json" in response_text:
        start = response_text.find("```json") + len("```json")
        end = response_text.find("```", start)
        if end != -1:
            json_str = response_text[start:end].strip()
            try:
                parsed = json.loads(json_str)
                if "SQL" in parsed:
                    return parsed["SQL"].strip()
            except json.JSONDecodeError as e:
                print("⚠️ JSON decode error:", e)

    # Step 2: fallback to legacy `SQLQuery:` and ```sql``` if needed
    if "```sql" in response_text:
        start = response_text.find("```sql") + len("```sql")
        end = response_text.find("```", start)
        if end != -1:
            return response_text[start:end].strip()

    sql_query_start = response_text.find("SQLQuery:")
    if sql_query_start != -1:
        response_text = response_text[sql_query_start:]
        if response_text.startswith("SQLQuery:"):
            response_text = response_text[len("SQLQuery:") :]
    sql_result_start = response_text.find("SQLResult:")
    if sql_result_start != -1:
        response_text = response_text[:sql_result_start]

    return response_text.strip().strip("```").strip()

sql_parser_component = FnComponent(fn=parse_response_to_sql)

# Dynamically build schema string for prompt
schema_str = "\n".join([f"{t}: {table_names_and_columns[t]}" for t in table_names])

# ---- Dataset switch: selects both the text2sql and answer-synthesis prompts ----
DATASET = "ynhhqa"  # options: "ynhhqa" | "drugehrqa" | "ehrsql" | "ehrnoteqa"

TEXT2SQL_PROMPTS = {
    "ehrsql": """Given an input question, generate a syntactically correct {dialect} query that directly answers it using only the tables and columns defined in the provided schema. The query must include all necessary calculations, conditions, and logical operations required to produce a complete and correct result.
Never query for all columns from a specific table; only select the columns that are relevant to the question.
Pay close attention to use only the column names visible in the schema description. 
Be careful not to query columns that do not exist. 
Ensure each column is used from the correct table and qualify column names with their table name when necessary to avoid ambiguity. 

Instructions:
1. Time-based reasoning:
   - Perform all time-related calculations directly in SQL.
   - Use days as the unit of time rather than hours.
   - When identifying the first or last event, sort by the start time column (not the end time).
2. Boolean existence checks:
   - For yes/no questions return a Boolean-like result.
3. Schema adherence:
   - Use only column names and tables present in the schema.
   - Qualify columns with their table name if needed to avoid ambiguity.
4. Column semantics:
   - Differentiate correctly between admission location and admission type based on the question.
5. Ordering for interpretability:
   - If ordering improves clarity, apply ORDER BY on the most relevant column (typically time or numeric values).
6. Not every question can be answered from this database. Before writing a query, study the table information given above, what each table is, what its columns actually hold, and what the sample content shows, and from that work out the scope of this database. If nothing in these tables holds that kind of information, output exactly 'null'.
7. Output format:
   - Only output the final SQL query.
   - Do not include any explanations, comments, or additional text.

Following this format as output:
Question: Question here
SQLQuery: SQL query to run

Only use tables listed below.
{schema}

Question: {query_str}
SQLQuery:
""",
    "drugehrqa": """Given an input question, first create a syntactically correct {dialect} query to run. 
You can order the results by a relevant column to return the most informative or representative examples in the database.
Never query for all columns from a specific table; only select the columns that are relevant to the question.
Pay close attention to use only the column names visible in the schema description. 
Be careful not to query columns that do not exist. 
Ensure each column is used from the correct table and qualify column names with their table name when necessary to avoid ambiguity. 
Please include the full entity name and hadm_id in the SQL if there is any in the question.

Following this format as output:
Question: Question here
SQLQuery: SQL query to run

Only use tables listed below.
{schema}

Question: {query_str}
SQLQuery:
""",
    "ynhhqa": """Given an input question, generate a syntactically correct {dialect} SQL query that directly answers it.
If applicable, order the results by a relevant column (e.g., time or numeric value) to return the most informative examples in the database.

Please include all relevant tables that could potentially provide useful information.
Carefully identify the key entity mentioned in the question and accurately select the table where it would reside.
Use only the column names present in the schema description—do not reference columns that do not exist.
Always ensure that columns are correctly associated with their corresponding tables and qualify them with table names when needed to avoid ambiguity.
The SQL query must be generated within a single line following the exact format below.

Follow these additional instructions:
1. Evaluate whether the entity mentioned in the question exists in the EHR data. If uncertain, query all values from the relevant column.
2. Review all tables carefully and include every table that may be relevant to the question.
3. When searching for entity names (e.g., drugs, conditions), use the source value columns and apply `LIKE` instead of `=` for matching. Use lowercase for comparison.
4. Expand each key entity (drug name, condition, hospital stay type, etc.) with all possible variations (abbreviations, short forms, full names) using `OR` conditions for robust search.
5. Use `JOIN` only when necessary to resolve concept names from concept ID tables, not to combine unrelated tables such as drugs and labs.
6. Always include the relevant time column in your query, as temporal information is crucial for clinical decision support.
7. Include lab, drug, or other relevant tables that could help answer the question, when appropriate.
8. Do not use the NOTE table in any query.
9. For visit types, resolve the visit concept ID via the concept table and use the concept name to identify specific visit types.
10. Return the output strictly with the following structure:

Following this format as output:
Question: Question here
SQLQuery: SQL query to run

Only use tables listed below.
{schema}

Question: {query_str}
SQLQuery:
""",
}

# EHRNoteQA is notes-centric; it reuses a generic SQL prompt so the pipeline runs
# normally (structured result may simply be none). Its synthesis prompt is set below.
TEXT2SQL_PROMPTS["ehrnoteqa"] = TEXT2SQL_PROMPTS["drugehrqa"]

text2sql_prompt = DEFAULT_TEXT_TO_SQL_PROMPT.partial_format(
    dialect=engine.dialect.name
)
text2sql_prompt.template = TEXT2SQL_PROMPTS[DATASET]

In [ ]:
def debug_function(input_data):
    print("DebugComponent Output:")
    print(input_data)
    return input_data

# Create a FnComponent for debugging
debug_component = FnComponent(fn=debug_function)

In [ ]:
# ---- Answer-synthesis prompts, selected by the same DATASET switch ----
SYNTH_PROMPTS = {
    "drugehrqa": """Given an input question, answer the question based on the query results.
You should also output the given SQL query and notes information. Only answer with the returned information.
Please pay attention to the temporal sequence and rethink the time influence when you answer the question.
If the time in notes does not match time in the question, ignore them.
Do not make up something.
Please provide as much detail as possible when you answer the question.

Input:
- Query: {query_str}
- SQL Query: {sql_query}
- SQL Response: {context_str}
- Notes: {notes}

Output:
1. SQL QUERY: Provide the SQL query used.
2. Evidence from notes: Provide some information from notes.
3. Response: Provide a concise and accurate response to the question based on the SQL response and notes, please include all the necessary details from the evidence to support your answer for the question.
""",
    "ynhhqa": """Given an input question, answer it based on the executed SQL results and the retrieved clinical notes.

Instructions:
1. Synthesize the Answer: Combine the quantitative facts from the SQL results (e.g., lab values, medication times) with the qualitative context from the notes.
2. Temporal Reasoning: Pay strict attention to the timeline. Ensure that the events described fall within the specific time window or encounter requested in the question (e.g., "last ED visit", "within 24 hours").
3. Conflict Resolution: If the notes and tables conflict, prioritize the structured SQL results for numerical values (dosages, timestamps) and notes for clinical status.
4. Formatting: Provide a concise, factual answer.

Input Context:
- User Question: {query_str}
- Executed SQL Query: {sql_query}
- Structured Data Evidence (SQL Result): {context_str}
- Unstructured Data Evidence (Clinical Notes): {notes}

Output:
1. Reasoning: Briefly explain how you derived the answer from the evidence.
2. Final Answer: The direct answer to the user's question.
""",
    # EHRNoteQA: notes-based summary. SQL vars are kept only so the shared pipeline
    # can pass them (may be none); the answer is derived from the notes.
    "ehrnoteqa": """You are a highly knowledgeable assistant specializing in answering medical questions.
Please answer the question based on the context in one sentence.
CONTEXT: {notes}
Question: {query_str}
(Structured evidence, if any -- SQL: {sql_query}; SQL result: {context_str})
OUTPUT:
""",
}

# EHRSQL has no dedicated multimodal synthesis prompt -> fall back to DrugEHRQA's.
_synth_key = DATASET if DATASET in SYNTH_PROMPTS else "drugehrqa"
response_synthesis_prompt = PromptTemplate(SYNTH_PROMPTS[_synth_key])

In [ ]:
# SQL execution with iterative error-correction (paper: regenerate SQL on failure)
def _extract_sql(text_out: str) -> str:
    t = text_out.strip()
    if "```sql" in t:
        t = t.split("```sql")[1].split("```")[0]
    elif "```" in t:
        t = t.split("```")[1].split("```")[0]
    return t.strip().rstrip(";").strip()

def run_sql_with_retry(sql_query: str, query_str: str = "", max_retries: int = 3) -> str:
    last_err = ""
    for _ in range(max_retries):
        try:
            with engine.connect() as conn:
                rows = conn.execute(text(sql_query)).fetchall()
            return str([tuple(r) for r in rows])
        except Exception as e:
            last_err = str(e)
            fix = Settings.llm.complete(
                f"This {engine.dialect.name} SQL failed with error: {last_err}\n"
                f"SQL: {sql_query}\nQuestion: {query_str}\n"
                f"Return only the corrected SQL, no explanation."
            ).text
            sql_query = _extract_sql(fix)
    return f"SQL execution failed after {max_retries} tries: {last_err}"

sql_exec_component = FnComponent(fn=run_sql_with_retry)

qp = QP(
    verbose = True,
)
Settings.callback_manager = qp.callback_manager

qp.add_modules({
    "input": InputComponent(),
    "table_retriever": obj_retriever,
    "note_retriever": note_retriever_component,
    "table_output_parser": table_parser_component,
    "text2sql_prompt": text2sql_prompt,
    "text2sql_llm": Settings.llm,
    "debug_component": debug_component,
    "sql_output_parser": sql_parser_component,
    "sql_exec": sql_exec_component,
    "response_synthesis_prompt": response_synthesis_prompt,
    "response_synthesis_llm": Settings.llm,
    "debug_function": debug_component,
})

qp.add_link("input", "table_retriever")
qp.add_link("table_retriever", "table_output_parser", dest_key="selected_tables")
qp.add_link("input", "text2sql_prompt", dest_key="query_str")
qp.add_link("table_output_parser", "text2sql_prompt", dest_key="schema")
qp.add_chain(["text2sql_prompt", "text2sql_llm", 'debug_component', "sql_output_parser"])

# SQL execution + retry
qp.add_link("sql_output_parser", "sql_exec", dest_key="sql_query")
qp.add_link("input", "sql_exec", dest_key="query_str")

qp.add_link("sql_output_parser", "response_synthesis_prompt", dest_key="sql_query")
qp.add_link("sql_exec", "response_synthesis_prompt", dest_key="context_str")

qp.add_link("input", "note_retriever", dest_key="inputs")
qp.add_link("sql_exec", "note_retriever", dest_key="structured_results")

qp.add_link("note_retriever", "response_synthesis_prompt", dest_key="notes")
qp.add_link("input", "response_synthesis_prompt", dest_key="query_str")
qp.add_link("response_synthesis_prompt", "debug_function")
qp.add_link("debug_function", "response_synthesis_llm")

In [ ]:
import datetime

query = f"""How did the INR change in the days following heparin administration for patient 123, and what is the time for every drugs and labs?
"""

responses = []
count = 0
start_time = datetime.datetime.now()   # Record start time

while len(responses) < 1 and count < 5:
    count += 1
    try:
        response = qp.run(
            query=query
        )
        responses.append(response.message.content)
        print(f"\n\nSYSTEM: \n{str(response)}")
    except Exception:
        continue

end_time = datetime.datetime.now()     # Record end time
elapsed = end_time - start_time        # Calculate elapsed time

if len(responses) == 0 and count == 5:
    print('Cannot generate correct SQL Query based on that question. Would you please refine the question?')

# Print time at the end
# print(f"\nStart Time: {start_time}")
# print(f"End Time:   {end_time}")
print(f"Elapsed:    {elapsed}")